# Purpose Insert Data from API to Google Sheet and Read Data to Data Frame from Google Sheet

# 📋 Steps to Insert Data into Google Sheet Using Python

## 1. Enable Google Sheets API
- Visit [Google Cloud Console](https://console.cloud.google.com/)
- Create a new project or select an existing one
- Enable the following APIs:
  - Google Sheets API
  - Google Drive API

## 2. Create Service Account Credentials
- Navigate to **IAM & Admin > Service Accounts**
- Create a new service account
- Generate and download the JSON key file
- Share your target Google Sheet with the service account’s email address

## 3. Install Required Python Libraries
- Use pip to install dependencies:
  - `gspread`
  - `google-auth`  
  *!pip install gspread*  
*!pip install google-auth*  

In [ ]:
import gspread
print("gspread is working!")


## 4. Authenticate and Connect to Google Sheets
- Load the service account credentials from the JSON file
- Define scopes for Sheets and Drive access
- Authorize the client using `gspread`
- Open the target spreadsheet and select the desired worksheet


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## 5. Prepare the Data
- Structure your data as a list or dictionary
- Ensure all values are JSON-serializable
  - Convert `datetime` objects to strings if needed

Check if **gspread** is working

In [ ]:
from getpass import getpass

api_key = getpass("Enter your NewsAPI key: ")
# eca9a30d27174c558c4fb34d84185d59

In [ ]:
from datetime import datetime, timedelta

# Current timestamp
now = datetime.now()

# Subtract 15 hours for from_range
from_range = now - timedelta(hours=48)

# Subtract 2 hours for to_range
to_range = now - timedelta(hours=24)

# Format as string
from_range_str = from_range.strftime("%Y-%m-%d %H:%M:%S")
to_range_str = to_range.strftime("%Y-%m-%d %H:%M:%S")

print("From Range:", from_range_str)
print("To Range:", to_range_str)

In [ ]:
from google.oauth2.service_account import Credentials
scope = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]
creds = Credentials.from_service_account_file(
    "/kaggle/input/cred-file/sapient-ground-454808-v8-1e8cd4e1d313.json",
    scopes=scope
)
client = gspread.authorize(creds)
sheet_names = [ws.title for ws in client.open("share_sheet").worksheets()]
print("Sheet names:", sheet_names)


## 6. Insert Data into the Sheet
- Use `append_row()` to add data to the next available row
- Optionally use `update()` or `insert_row()` for more control

In [ ]:
import requests
import gspread
from datetime import datetime

# Step 1: Fetch News Data
# news_url = "https://newsapi.org/v2/top-headlines"
# params = {
#     "country": "us",
#     "category": "business",
#     "apiKey": "eca9a30d27174c558c4fb34d84185d59",
#     "from": "2025-10-04",
#     "to": "2025-10-05",
#     "sortBy": "popularity"
# }

news_url = "https://newsapi.org/v2/everything"
params = {
    "q": "news",
    "from": from_range_str,
    "to": to_range_str,
    "sortBy": "popularity",
    "language": "en",
    "apiKey": api_key
}

response = requests.get(news_url, params=params)
articles = response.json().get("articles", [])

# Step 2: Setup Google Sheets
scope = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]
creds = Credentials.from_service_account_file(
    "/kaggle/input/cred-file/sapient-ground-454808-v8-1e8cd4e1d313.json",
    scopes=scope
)
client = gspread.authorize(creds)
sheet = client.open("share_sheet").sheet1

# Step 3: Track sequence numbers
source_seq = {}           # Maps source to fixed sequence number
source_author_count = {}  # Maps (source, author) to sub-sequence count

# Step 4: Insert each article into the sheet
all_rows = []

for article in articles:
    source = article.get("source", {}).get("name") or "Unknown Source"
    author = article.get("author") or "Unknown Author"
    title = article.get("title") or "Untitled"
    # content = article.get("content") or "No content available"
    content = article.get("description") or article.get("content") or "No content available"
    published_at = article.get("publishedAt") or ""

    try:
        formatted_published_at = datetime.strptime(published_at, "%Y-%m-%dT%H:%M:%SZ").strftime("%Y-%m-%d %H:%M:%S")
    except ValueError:
        formatted_published_at = published_at

    if source not in source_seq:
        source_seq[source] = len(source_seq) + 1

    key = (source, author)
    source_author_count[key] = source_author_count.get(key, 0) + 1

    row = [
        source_seq[source],
        source_author_count[key],
        source,
        author,
        title,
        article.get("description"),
        content,
        True,
        formatted_published_at,
    ]

    all_rows.append(row)

# ✅ Write all rows at once
sheet.append_rows(all_rows)

In [ ]:
news_url = "https://newsapi.org/v2/everything"
params = {
    "q": "news",
    "from": from_range_str,
    "to": to_range_str,
    "sortBy": "popularity",
    "language": "es",
    "apiKey": api_key
}

response = requests.get(news_url, params=params)
articles = response.json().get("articles", [])

# Step 2: Setup Google Sheets
scope = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]
creds = Credentials.from_service_account_file(
    "/kaggle/input/cred-file/sapient-ground-454808-v8-1e8cd4e1d313.json",
    scopes=scope
)
client = gspread.authorize(creds)
sheet = client.open("share_sheet").worksheet("Sheet2")

# Step 3: Track sequence numbers
source_seq = {}           # Maps source to fixed sequence number
source_author_count = {}  # Maps (source, author) to sub-sequence count

# Step 4: Insert each article into the sheet
all_rows = []

for article in articles:
    source = article.get("source", {}).get("name") or "Unknown Source"
    author = article.get("author") or "Unknown Author"
    title = article.get("title") or "Untitled"
    # content = article.get("content") or "No content available"
    content = article.get("description") or article.get("content") or "No content available"
    published_at = article.get("publishedAt") or ""

    try:
        formatted_published_at = datetime.strptime(published_at, "%Y-%m-%dT%H:%M:%SZ").strftime("%Y-%m-%d %H:%M:%S")
    except ValueError:
        formatted_published_at = published_at

    if source not in source_seq:
        source_seq[source] = len(source_seq) + 1

    key = (source, author)
    source_author_count[key] = source_author_count.get(key, 0) + 1

    row = [
        source_seq[source],
        source_author_count[key],
        source,
        author,
        title,
        article.get("description"),
        content,
        True,
        formatted_published_at,
    ]

    all_rows.append(row)

# ✅ Write all rows at once
sheet.append_rows(all_rows)

## 7. Verify the Insertion
- Open the Google Sheet in your browser
- Confirm that the data appears correctly in the expected location

# Sheet 1  
## code using Pandas

In [ ]:
import pandas as pd

# Fetch all records from the sheet
records = client.open("share_sheet").worksheet("Sheet1").get_all_records()

# Convert to DataFrame
df_sheet1 = pd.DataFrame(records)

# Preview the data
print(df_sheet1.head(5))


# Sheet 2  
## code using pyspark

In [ ]:
import pandas as pd
from pyspark.sql import SparkSession


# Fetch all records from the sheet
records = client.open("share_sheet").worksheet("Sheet2").get_all_records()

# Convert to DataFrame
pandas_df = pd.DataFrame(records)

spark = SparkSession.builder.appName("PandasToSparkConversion").getOrCreate()

df_sheet2 = spark.createDataFrame(pandas_df)
# Preview the data
df_sheet2.show(5)
df_sheet2.printSchema()


### Missing Value Detection  
#### Sheet 1 Pandas

In [ ]:
# Column-wise missing value count
print("Missing values per column:")
print(df_sheet1.isnull().sum())

# Rows with any missing values
print("\nRows with missing data:")
print(df_sheet1[df_sheet1.isnull().any(axis=1)])

### Missing Value Detection  
#### Sheet 2 pySpark

In [ ]:
from pyspark.sql.functions import col, when, count, isnan, sum

# Column-wise missing value count
missing_counts = df_sheet2.select([
    count(when(col(c).isNull() | isnan(c), c)).alias(c)
    for c in df_sheet2.columns
])
missing_counts.show()

# Start with a neutral condition
condition = col(df_sheet2.columns[0]).isNull() | isnan(df_sheet2.columns[0])

# Chain the rest using OR
for c in df_sheet2.columns[1:]:
    condition = condition | col(c).isNull() | isnan(c)

# Apply filter
df_sheet2.filter(condition).show()


### Primary Key Uniqueness Validation  
#### Sheet 1 panda

In [ ]:
# Define primary key column(s)
primary_key_cols = ['div_heading', 'sub_div_heading']  # Adjust based on your dataset

# Check for duplicates
duplicates = df_sheet1[df_sheet1.duplicated(subset=primary_key_cols, keep=False)]

if duplicates.empty:
    print("✅ Primary key is unique.")
else:
    print("❌ Duplicate primary key values found:")
    print(duplicates)

### Primary Key Uniqueness Validation  
#### Sheet 2 pyspark

In [ ]:
# Define primary key column(s)
primary_key_cols = ['div_heading', 'sub_div_heading']  # Adjust based on your dataset

# Group by primary key and count occurrences
pk_counts = df_sheet2.groupBy(primary_key_cols).agg(count("*").alias("count")).filter("count > 1")

if pk_counts.count() == 0:
    print("✅ Primary key is unique.")
else:
    print("❌ Duplicate primary key values found:")
    pk_counts.show()

### Duplicate Record Identification
#### Sheet 1 pandas

In [ ]:
full_duplicates = df_sheet1[df_sheet1.duplicated(keep=False)]
print("🔍 Full-row duplicates:")
print(full_duplicates)

# Detect duplicates based on specific columns
subset_cols = ["div_heading", "sub_div_heading"]
partial_duplicates = df_sheet1[df_sheet1.duplicated(subset=subset_cols, keep=False)]
print("\n🔍 Duplicates based on div_heading and sub_div_heading:")
print(partial_duplicates)

### Duplicate Record Identification
#### Sheet 2 spark

In [ ]:
from pyspark.sql.functions import count

columns = ['div_heading', 'sub_div_heading', 'contents_head', 'details', 'comments_ifany']

# Detect full-row duplicates
df_sheet2.groupBy(columns).agg(count("*").alias("count")).filter("count > 1").show()

# Detect duplicates based on subset of columns
subset_cols = ['div_heading','sub_div_heading','contents_head']
df_sheet2.groupBy(subset_cols).agg(count("*").alias("count")).filter("count > 1").show()

 